In [ ]:
import pickle
from tqdm import tqdm
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
from transformers import AutoFeatureExtractor, ResNetModel, AutoTokenizer, AutoModel
from transformers import ViTFeatureExtractor, ViTModel
from typing import List, Dict
import numpy as np
import os
import pandas as pd
from transformers import pipeline
from PIL import Image
import itertools
import gc

In [ ]:

def chunkify_primary(embs,nft2txs, by):
    """Input:
            dizionario contenente per ogni nft la lista delle sue transazioni 
            metodo che segmenta le transazioni per unità temporali (mensile, trimestrale...)
        
        Output:
            dizionario che contiene, per ogni nft la lista DEI SOLI nft comparsi in quello snapshot
        """
    chunks = {}

    for nft in tqdm(embs, "Generazione chunks"):
        nft_data = nft2txs[nft]
        for tx in nft_data["txs"]:
            segmented_time = by(tx["timestamp"])
            if not segmented_time in chunks:
                chunks[segmented_time] = {}
            if nft not in chunks[segmented_time]:
                chunks[segmented_time][nft] = [tx]
            break
    return {k: v for k, v in sorted(chunks.items(), key=lambda x: x[0])}


def chunkify_all(embs,nft2txs, by):
    """Input:
            dizionario contenente per ogni nft la lista delle sue transazioni 
            metodo che segmenta le transazioni per unità temporali (mensile, trimestrale...)
        
        Output:
            dizionario che contiene, per ogni nft la lista DI TUTTI gli nft comparsi fino a quello snapshot
        """
    chunks = {}

    for nft in tqdm(embs, "Generazione chunks"):
        nft_data = nft2txs[nft]
        for tx in nft_data["txs"]:
            segmented_time = by(tx["timestamp"])
            if not segmented_time in chunks:
                chunks[segmented_time] = {}
            if nft not in chunks[segmented_time]:
                chunks[segmented_time][nft] = [tx]
            else:
                chunks[segmented_time][nft].append(tx)
    return {k: v for k, v in sorted(chunks.items(), key=lambda x: x[0])}


def loadEmbs(path):
    """"Carica gli embedding nft-aware da un path"""
    EMBS = {
        "train": torch.load(f'{path}/nft2emb_train.pt', map_location="cpu"),
        "val": torch.load(f'{path}/nft2emb_val.pt', map_location="cpu"),
    }
    return EMBS


In [ ]:
path = "/home/user/Progetti/NFT/BASELINE_FOR_ABLATION"
path_images = "../input/bigImageDataset/imgs"

id2img = pickle.load(open("../embeddings/unique2path.pkl", "rb"))


def openImage(path):
    return Image.open(path).convert('RGBA').convert('RGB')


split = "train"
device = "cuda"


# feature_extractor = AutoFeatureExtractor.from_pretrained(
#     "google/vit-base-patch16-224")
# model = AutoModel.from_pretrained("google/vit-base-patch16-224").to(device)
# model.eval()


@torch.no_grad()
def extract_feature(image, pool_index=0):
    """Genera le feature facendo CLS-pooling"""
    inputs = feature_extractor(image, return_tensors="pt")
    for k in inputs:
        inputs[k] = inputs[k].to(device)
    outputs = model(**inputs)
    pooled_output = outputs.last_hidden_state[:, pool_index].reshape(-1)
    model.zero_grad()
    del outputs
    del inputs
    return pooled_output


nft2txs = pickle.load(open(f"../input/nft2txs.pkl", "rb"))


nft2emb = loadEmbs(f"{path}/vit-base-patch16-224")[split]

nft2category = pickle.load(open(f"../input/nft2category.pkl", "rb"))
coll2cat = {}
for nft in nft2emb:
    if eval(nft)[0] not in coll2cat:
        coll2cat[eval(nft)[0]] = nft2category[nft]

nft2emb_aware = {k: torch.from_numpy(v).cpu() for k, v in nft2emb.items()}

if os.path.exists(f"../embeddings/nft2emb_dummyViT-{split}.pkl"):
    nft2emb_unaware = pickle.load(
        open(f"../embeddings/nft2emb_dummyViT-{split}.pkl", 'rb'))
else:
    nft2emb_unaware = {k: extract_feature(openImage(f'{path_images}/{id2img[k]}')).cpu(
    ) for k, v in tqdm(nft2emb.items(), "Running embeddings generation for vit")}
    pickle.dump(nft2emb_unaware, open(
        f"../embeddings/nft2emb_dummyViT-{split}.pkl", 'wb'))

# coll2date=pickle.load(open("../input/coll2created.pkl","rb"))
# coll2date={k:v for k,v in coll2date.items() if v is not None}
# print(len(nft2emb_aware))
# nft2emb_unaware={k:v for k,v in nft2emb_unaware.items() if eval(k)[0] in coll2date}
# nft2emb_aware={k:v for k,v in nft2emb_aware.items() if eval(k)[0]  in coll2date}
# print(len(nft2emb_aware))


In [ ]:
d = nn.CosineSimilarity(dim=1)
import torchmetrics
def getAllDistances(embs):
    DISTS=np.zeros(shape=(len(embs),len(embs)),dtype=np.float16)
    BATCH_SIZE=2048
    PRECISION=4
    embs_vector = torch.cat([embs[nft].unsqueeze(0)
                              for nft in embs]).to(device,dtype=torch.float16)
    embs_key = np.array([nft for nft in embs])       
    
    NFT_LIST=[]
    EMB_LIST=[]   
    N=0
    for i,(n,e) in enumerate(tqdm(embs.items())):
        NFT_LIST.append(n)
        EMB_LIST.append(e)
        if len(NFT_LIST)>=BATCH_SIZE or N+len(NFT_LIST)==len(embs):
            emb=embs_vector[N:N+len(NFT_LIST)]
            dists=torchmetrics.functional.pairwise_cosine_similarity(embs_vector, emb).cpu().numpy().astype(np.float16).T
            #rendo diagonale per risparimare memoria, tanto j,i non serve
            for j in range(len(NFT_LIST)):
                dists[j,len(embs)-(N+j):]=-1
                
            DISTS[N:N+len(NFT_LIST),:]=np.round(dists,PRECISION)
            N+=len(NFT_LIST)
            EMB_LIST=[]
            NFT_LIST=[]

    return DISTS,{nft:i for i,nft in enumerate(embs_key)}
def getPDist(x1,x2):
    # x1 = torch.cat([embs1[nft].unsqueeze(0)
    #                           for nft in embs1]).to(device,dtype=torch.float16)
    # x2 = torch.cat([embs2[nft].unsqueeze(0)
    #                           for nft in embs2]).to(device,dtype=torch.float16)

    DISTS=torch.zeros(size=(len(x1),len(x2)))
    BATCH_SIZE=2048

    EMB_LIST=[]   
    N=0
    for i,(e) in enumerate(x1):
        EMB_LIST.append(e)
        if len(EMB_LIST)>=BATCH_SIZE or N+len(EMB_LIST)==len(x1):
            emb=x1[N:N+len(EMB_LIST)]
            dists=torchmetrics.functional.pairwise_cosine_similarity(x2, emb).T
            DISTS[N:N+len(EMB_LIST),:]=dists.cpu()
            N+=len(EMB_LIST)
            EMB_LIST=[]
    return DISTS

In [ ]:
NFT2SEEN={}
COLL2SEEN={}

for nft in tqdm(nft2emb_unaware):
    txs=nft2txs[nft]
    timestamps=[tx["timestamp"] for tx in txs["txs"]]
    NFT2SEEN[nft]=timestamps[np.argmin(timestamps)]
COLLS=list(set([eval(nft)[0] for nft in NFT2SEEN]))

COLL2NFTS={}
for nft in NFT2SEEN:
    if eval(nft)[0] not in COLL2NFTS:
        COLL2NFTS[eval(nft)[0]]=[]
    COLL2NFTS[eval(nft)[0]].append(nft)

for coll in tqdm(COLL2NFTS):
    timestamps=[NFT2SEEN[nft] for nft in COLL2NFTS[coll]]
    COLL2SEEN[coll]=timestamps[np.argmin(timestamps)]

In [ ]:
# if not os.path.isfile("../input/PAIRWISE_NFT_DISTANCES.csv"):
#     DISTS,embs2key=getAllDistances(nft2emb_unaware)
#     # pd.DataFrame(
    
#     # DISTS,
#     # index=      list(embs2key.keys()),
#     # columns=    list(embs2key.keys())
#     # ).to_csv("../input/PAIRWISE_NFT_DISTANCES.csv",chunksize=1000000)

# else:
#     DISTS=pd.read_csv("../input/PAIRWISE_NFT_DISTANCES.csv",index_col=0)
#     embs2key={e:i for  i,e in enumerate(DISTS.index.values)}
#     DISTS=DISTS.values

# def getDist(DISTS,embs2key,nft1,nft2):
#    id_nft1=embs2key[nft1]
#    id_nft2=embs2key[nft2]
#    d=DISTS[id_nft1][id_nft2]
#    if d==-1:
#       return DISTS[id_nft2][id_nft1]
#    else:
#       return d 


In [ ]:
def segment(timestamp):
    # return str((timestamp.year,(timestamp.week-1)))
    return str((timestamp.year, (timestamp.month-1)//1))
def coll(nft):
    return eval(nft)[0]
NFT2SEEN_SEG={k:segment(v) for k,v in NFT2SEEN.items()}
COLL2SEEN_SEG={k:segment(v) for k,v in COLL2SEEN.items()}

snapshots = chunkify_primary(nft2emb_unaware,nft2txs, by=segment)
snapshots_all = chunkify_all(nft2emb_unaware,nft2txs, by=segment)


seg = "Monthly"

s = list(snapshots.keys())
s = sorted(s, key=lambda x: eval(x)[0]*100+eval(x)[1])
SNAPSHOT2SNAPSHOTIDX={snap:i for i,snap in enumerate(s)}

In [ ]:
def mean_fn(tensor):
    return torch.mean(tensor)
def max_fn(tensor):
    return torch.max(tensor,axis=1).values
def min_fn(tensor):
    return torch.min(tensor,axis=1).values

def COLL_LevelGraph(snapshot,old_metadata={},coll2nfts={},pool_fn=max_fn,aggreg_fn=mean_fn):
    graph={"E": [], "V": []}
    #definisco la funzione di similarità 
    d_fn = nn.CosineSimilarity(dim=1)
    #dizionario che contiene per ogni collection la lista delle transazioni 
    new_metadata={}
    
    #coll2nfts contiene per ogni collection la lista degli nft con una primary sell incontrati finora, per semplicità vengono salvati come 
    #una lista numpy quindi qui li riconverto in lista
    for coll in coll2nfts:
        coll2nfts[coll]=list(coll2nfts[coll])
    #ci aggiungo gli nft di questo snapshot se non erano già presenti
    for nft in snapshot:
        if eval(nft)[0] not in coll2nfts:
            coll2nfts[eval(nft)[0] ]=[]
        coll2nfts[eval(nft)[0]].append(nft)
    #ritransformo in numpy    
    for coll in coll2nfts:
        coll2nfts[coll]=np.array(coll2nfts[coll])
    #a questo punto costruisco un dizionario che contiene come chiave la collection e come valori la lista di "first-selling-time"
    #di tutti gli nft con una primary sell fino a questo snapshot
    coll2nftseen={}    
    for coll,nfts in coll2nfts.items():
        coll2nftseen[coll]=np.array([SNAPSHOT2SNAPSHOTIDX[NFT2SEEN_SEG[nft]] for nft in nfts])
    #popolo ora new_metadata che conterrà per ogni collection la lista dei prezzi delle transazioni di tutti gli nft con una primary sell fino a questo momento
    #prima ci aggiungo le transazioni "fino ad ora"
    for coll,metadata in old_metadata.items():
        new_metadata[coll]=metadata
    # poi ci aggiungo le transazioni di "ora"
    for nft in snapshot:
        if eval(nft)[0] not in new_metadata:
            new_metadata[eval(nft)[0]]=[]
        new_metadata[eval(nft)[0]].extend([float(tx["price"]) for tx in snapshot[nft]])
            
    #NEW SNAPSHOT
    #per ogni coppia di collection
    for coll1,nfts1 in coll2nfts.items():

        for coll2,nfts2 in coll2nfts.items():
        
            #i soli nft che possono aver copiato da coll2 sono queli che sono nati dopo della nascita di coll2
            #prendo l'indice di prima sell   di un nft di coll2
            idx_seen_coll2=SNAPSHOT2SNAPSHOTIDX[segment(COLL2SEEN[coll2])]
            #prendo i soli nft di coll1 che rispettano la condizione TIME(coll1)>MINTED(coll2)
            nfts1_=nfts1[coll2nftseen[coll1]>idx_seen_coll2]
            #se ci sono degli nft e le collection c1,c2 sono diverse
            if len(nfts2)>0 and len(nfts1_)>0 and coll1!=coll2:
                #creo i due tensori contenenti per ogni colletion la lista degli embedding degli nft ivi contenuti
                embs_coll1=torch.cat([nft2emb_unaware[nft].unsqueeze(0)for nft in nfts1_]).to(device)
                embs_coll2=torch.cat([nft2emb_unaware[nft].unsqueeze(0)for nft in nfts2]).to(device)
                #genero le distanze tramite il metodo a batch 
                dists = getPDist(embs_coll1, embs_coll2)
                #applico prima il pooling e poi aggregazione MEAN(MAX) MEAN(MEAN) MEAN(MIN)
                d=aggreg_fn(pool_fn(dists)).item()
                c=len(nfts1_)
                nc=len(coll2nftseen[coll1])-len(nfts1_)
                if nc!=0:
                    scaling_factor=1/(1+np.exp(-(c/nc)))
                else:
                    scaling_factor=1
                weight=scaling_factor*d
                graph["E"].append({
                        "source":coll1,
                        "target":coll2,
                        "raw_similarity":d,
                        "size_source":len(coll2nftseen[coll1]),
                        "used_source":len(nfts1_),
                        "size_target":len(coll2nftseen[coll2]),
                        "used_target":len(nfts2),
                        "scaling_factor":scaling_factor,
                        "c":c,
                        "nc":nc,
                        "weight":weight
                        })
            graph["V"].append(coll1)
            graph["V"].append(coll2)

    graph["V"]=list(set(graph["V"]))
    for coll in new_metadata:
        new_metadata[coll]=list(np.nan_to_num(new_metadata[coll]))
        new_metadata[coll]=[tx for tx in new_metadata[coll] if tx>0]
        if len(new_metadata[coll])==0:
            new_metadata[coll]=[0]

    METADATA=[{
            "id":coll,
            "fisrt_seen":COLL2SEEN[coll],
            "first_seen_snap":segment(COLL2SEEN[coll]),
            "#tx":len(new_metadata[coll]),
            "avg_price":np.mean(new_metadata[coll]),
            "max_price":np.max(new_metadata[coll]),
            "min_price":np.min(new_metadata[coll]),
            "std_price":np.std(new_metadata[coll]),
            "volume":   np.sum(new_metadata[coll]),
            "collection":coll,
            "category":coll2cat[coll]
            } 
        for coll in new_metadata]
    return graph,new_metadata,METADATA,coll2nfts

In [ ]:
meta__={}
coll2nfts={}
pbar=tqdm(s)
FOLDER="../graphsV2/COLLLevel-MAX/"
for snapshot_id in pbar:
    os.makedirs(f"{FOLDER}/{snapshot_id}",exist_ok=True)
    snapshot =snapshots_all[snapshot_id]
    graph,meta__,metadata,coll2nfts=COLL_LevelGraph(snapshot,old_metadata=meta__,coll2nfts=coll2nfts,pool_fn=max_fn)
    edges=graph["E"]
    pd.DataFrame().from_records(edges).to_csv(f"{FOLDER}/{snapshot_id}/edge_list.csv")
    pd.DataFrame().from_records(metadata).to_csv(f"{FOLDER}/{snapshot_id}/metadata.csv")
    pbar.set_description(f'Snapshot {snapshot_id} |V_all|={len(metadata)} |V_graph|={len(graph["V"])} |E|={len(graph["E"])}')


In [ ]:
FOLDER="../graphsV2/COLLLevel-MAX/"
pbar=tqdm(s)

for snapshot_id in pbar:
    try:
        edge_list=pd.read_csv(f"{FOLDER}/{snapshot_id}/edge_list.csv",index_col=0)
        edge_list=edge_list[edge_list.weight>.5]
        edge_list.to_csv(f"{FOLDER}/{snapshot_id}/edge_list(0.5).csv"   ,index=False)
    except:
        pass

In [ ]:
meta__={}
coll2nfts={}
pbar=tqdm(s)
FOLDER="../graphsV2/COLLLevel-MIN/"
for snapshot_id in pbar:
    os.makedirs(f"{FOLDER}/{snapshot_id}",exist_ok=True)
    snapshot =snapshots_all[snapshot_id]
    graph,meta__,metadata,coll2nfts=COLL_LevelGraph(snapshot,old_metadata=meta__,coll2nfts=coll2nfts,pool_fn=min_fn)
    edges=graph["E"]
    pd.DataFrame().from_records(edges).to_csv(f"{FOLDER}/{snapshot_id}/edge_list.csv")
    pd.DataFrame().from_records(metadata).to_csv(f"{FOLDER}/{snapshot_id}/metadata.csv")
    pbar.set_description(f'Snapshot {snapshot_id} |V_all|={len(metadata)} |V_graph|={len(graph["V"])} |E|={len(graph["E"])}')


In [ ]:
FOLDER="../graphsV2/COLLLevel-MIN/"
pbar=tqdm(s)

for snapshot_id in pbar:
    try:
        edge_list=pd.read_csv(f"{FOLDER}/{snapshot_id}/edge_list.csv",index_col=0)
        edge_list=edge_list[edge_list.weight>.5]
        edge_list.to_csv(f"{FOLDER}/{snapshot_id}/edge_list(0.5).csv"   ,index=False)
    except:
        pass

In [ ]:
meta__={}
coll2nfts={}
pbar=tqdm(s)
FOLDER="../graphsV2/COLLLevel-MEAN/"
for snapshot_id in pbar:
    os.makedirs(f"{FOLDER}/{snapshot_id}",exist_ok=True)
    snapshot =snapshots_all[snapshot_id]
    graph,meta__,metadata,coll2nfts=COLL_LevelGraph(snapshot,old_metadata=meta__,coll2nfts=coll2nfts,pool_fn=mean_fn)
    edges=graph["E"]
    pd.DataFrame().from_records(edges).to_csv(f"{FOLDER}/{snapshot_id}/edge_list.csv")
    pd.DataFrame().from_records(metadata).to_csv(f"{FOLDER}/{snapshot_id}/metadata.csv")
    pbar.set_description(f'Snapshot {snapshot_id} |V_all|={len(metadata)} |V_graph|={len(graph["V"])} |E|={len(graph["E"])}')

In [ ]:
FOLDER="../graphsV2/COLLLevel-MEAN/"
pbar=tqdm(s)

for snapshot_id in pbar:
    try:
        edge_list=pd.read_csv(f"{FOLDER}/{snapshot_id}/edge_list.csv",index_col=0)
        edge_list=edge_list[edge_list.weight>.5]
        edge_list.to_csv(f"{FOLDER}/{snapshot_id}/edge_list(0.5).csv"   ,index=False)
    except:
        pass